In [1]:
import sqlite3
import pandas as pd

In [2]:
db_path = "../data/raw/statcast.sqlite"

# 2. Query the SQLite master table specifically for VIEWS
query = "SELECT name FROM sqlite_master WHERE type='view' AND name NOT LIKE 'sqlite_%';"

# 3. Read it into pandas
with sqlite3.connect(db_path) as conn:
    views_df = pd.read_sql_query(query, conn)

print("--- Views in Statcast DB ---")
print(views_df)

--- Views in Statcast DB ---
                       name
0             statcast_2016
1             statcast_2017
2             statcast_2026
3             statcast_2019
4             statcast_2018
5             statcast_2025
6             statcast_2024
7             statcast_2023
8             statcast_2022
9             statcast_2021
10            statcast_2020
11  statcast_missed_strikes
12    statcast_missed_balls


In [3]:
query = "SELECT * FROM statcast_missed_balls"

''' 
Query details : 
CREATE VIEW IF NOT EXISTS statcast_missed_balls AS
SELECT 
    statcast.*,
    -- Horizontal check: standard MLB home plate width is 17 inches (~0.7083 feet on each side)
    CASE 
        WHEN plate_x NOT BETWEEN -0.7083 AND 0.7083
          OR plate_z NOT BETWEEN sz_bot AND sz_top
        THEN 1 
        ELSE 0 
    END AS is_out_strike_zone
FROM 
    statcast
WHERE 
    description = 'called_strike';
'''

with sqlite3.connect(db_path) as conn:
    view = pd.read_sql_query(query, conn)

parquet_path = "../data/processed/statcast_missed_balls.parquet"
view.to_parquet(parquet_path, index=False)

print(f"Saved {len(view):,} rows to {parquet_path}")



Saved 1,270,862 rows to ../data/processed/statcast_missed_balls.parquet


In [4]:
query = "SELECT * FROM statcast_missed_strikes"

''' 
Query details : 
CREATE VIEW IF NOT EXISTS statcast_missed_strikes AS
SELECT 
    statcast.*,
    -- Horizontal check: standard MLB home plate width is 17 inches (~0.7083 feet on each side)
    CASE 
        WHEN plate_x BETWEEN -0.7083 AND 0.7083 
         AND plate_z BETWEEN sz_bot AND sz_top 
        THEN 1 
        ELSE 0 
    END AS is_in_strike_zone
FROM 
    statcast
WHERE 
    description = 'ball';
'''

with sqlite3.connect(db_path) as conn:
    view = pd.read_sql_query(query, conn)

parquet_path = "../data/processed/statcast_missed_strikes.parquet"
view.to_parquet(parquet_path, index=False)

print(f"Saved {len(view):,} rows to {parquet_path}")

Saved 2,580,712 rows to ../data/processed/statcast_missed_strikes.parquet
